In [1]:
import sys
import importlib.util

print("Python version:", sys.version)
print("Python executable:", sys.executable)

packages = [
    "openai",
    "langchain",
    "langchain_openai",
    "sentence_transformers",
    "huggingface_hub",
]

print("\nPackage availability:")

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if available else 'Not installed'}")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable: c:\Users\ShZ\.virtualenvs\backend-ai-VRoIuNzk\Scripts\python.exe

Package availability:
openai: Installed
langchain: Not installed
langchain_openai: Not installed
sentence_transformers: Not installed
huggingface_hub: Installed


In [5]:
import os
import httpx

from dotenv import load_dotenv, find_dotenv

# Load environment variables
env_path = find_dotenv(usecwd=True)

if not env_path:
    raise RuntimeError(".env file not found.")

load_dotenv(env_path)

api_key = os.getenv("JINA_API_KEY")

if not api_key:
    raise RuntimeError("JINA_API_KEY is not configured.")

# Model configuration
MODEL = "jina-embeddings-v5-text-small"
API_URL = "https://api.jina.ai/v1/embeddings"

texts = [
    "What are the symptoms of diabetes?",
    "علائم بیماری دیابت چیست؟",
    "Diabetes can cause increased thirst and frequent urination.",
    "بیماری دیابت ممکن است باعث تشنگی زیاد و تکرر ادرار شود."
]

# Send embedding request
response = httpx.post(
    API_URL,
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": MODEL,
        "input": texts,
        "task": "text-matching",
        "dimensions": 1024,
    },
    timeout=60,
)

# Check response
response.raise_for_status()

result = response.json()

# Inspect embeddings
for item in result["data"]:
    embedding = item["embedding"]

    print(f"\nText index: {item['index']}")
    print("Dimensions:", len(embedding))
    print("First 5 values:", embedding[:5])
    print("Vector type:", type(embedding))

print("\nUsage:", result.get("usage"))
print("\nEmbedding test completed successfully!")


Text index: 0
Dimensions: 1024
First 5 values: [-0.00254328, 0.0518718, -0.03107836, -0.05768502, 0.00430402]
Vector type: <class 'list'>

Text index: 1
Dimensions: 1024
First 5 values: [-0.01516889, 0.04403871, -0.03669893, -0.03963484, 0.01431258]
Vector type: <class 'list'>

Text index: 2
Dimensions: 1024
First 5 values: [-0.05501245, 0.05654058, -0.0259781, 0.02540506, 0.01212948]
Vector type: <class 'list'>

Text index: 3
Dimensions: 1024
First 5 values: [-0.1043622, 0.06087795, -0.04137592, 0.04691028, 0.03320615]
Vector type: <class 'list'>

Usage: {'total_tokens': 73}

Embedding test completed successfully!


In [6]:
import numpy as np
import httpx

def get_embeddings(texts):
    response = httpx.post(
        "https://api.jina.ai/v1/embeddings",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL,
            "input": texts,
            "task": "text-matching",
            "dimensions": 1024,
            "normalized": True,
        },
        timeout=60,
    )

    response.raise_for_status()

    data = response.json()["data"]

    return [
        np.array(item["embedding"], dtype=np.float32)
        for item in data
    ]


def cosine_similarity(vec1, vec2):
    return float(np.dot(vec1, vec2))


test_pairs = [
    (
        "What are the symptoms of diabetes?",
        "علائم بیماری دیابت چیست؟",
        "Cross-language equivalent"
    ),
    (
        "Diabetes can cause increased thirst.",
        "دیابت ممکن است باعث تشنگی زیاد شود.",
        "Cross-language related"
    ),
    (
        "What are the symptoms of diabetes?",
        "How do I repair a broken car engine?",
        "Unrelated topics"
    ),
    (
        "High blood pressure can increase cardiovascular risk.",
        "فشار خون بالا می‌تواند خطر بیماری‌های قلبی را افزایش دهد.",
        "Cross-language medical"
    ),
]

all_texts = [
    text
    for pair in test_pairs
    for text in pair[:2]
]

vectors = get_embeddings(all_texts)

print("Total embeddings:", len(vectors))

for i, (_, _, description) in enumerate(test_pairs):
    score = cosine_similarity(
        vectors[i * 2],
        vectors[i * 2 + 1]
    )

    print(f"{description}: {score:.4f}")

Total embeddings: 8
Cross-language equivalent: 0.9009
Cross-language related: 0.7580
Unrelated topics: 0.3364
Cross-language medical: 0.9156


In [7]:
import langchain_jina

print("Module:", langchain_jina.__file__)
print("Available components:")
print(dir(langchain_jina))

Module: c:\Users\ShZ\.virtualenvs\backend-ai-VRoIuNzk\Lib\site-packages\langchain_jina\__init__.py
Available components:
['LateChunkEmbeddings', 'LateChunkQdrant', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'embeddings', 'vectorestores']


In [8]:
import os
import httpx

from langchain_core.embeddings import Embeddings


class JinaEmbeddingsAdapter(Embeddings):

    def __init__(
        self,
        api_key: str,
        model: str = "jina-embeddings-v5-text-small",
        dimensions: int = 1024,
    ):
        self.api_key = api_key
        self.model = model
        self.dimensions = dimensions
        self.api_url = "https://api.jina.ai/v1/embeddings"

        self.client = httpx.Client(
            timeout=60.0,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
        )

    def _embed(self, texts: list[str], task: str) -> list[list[float]]:

        if not texts:
            return []

        response = self.client.post(
            self.api_url,
            json={
                "model": self.model,
                "input": texts,
                "task": task,
                "dimensions": self.dimensions,
                "normalized": True,
            },
        )

        response.raise_for_status()

        data = response.json()["data"]

        data.sort(key=lambda item: item["index"])

        return [
            item["embedding"]
            for item in data
        ]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts, "retrieval.passage")

    def embed_query(self, text: str) -> list[float]:
        return self._embed([text], "retrieval.query")[0]

    def close(self):
        self.client.close()


# Initialize adapter
embeddings = JinaEmbeddingsAdapter(
    api_key=api_key,
)

print("Jina LangChain Adapter initialized successfully!")

Jina LangChain Adapter initialized successfully!


In [9]:
# Test documents
documents = [
    "Diabetes is a chronic disease that affects blood sugar levels.",
    "دیابت یک بیماری مزمن است که سطح قند خون را تحت تأثیر قرار می‌دهد.",
    "Hypertension is a condition characterized by high blood pressure.",
]

# Generate document embeddings
document_vectors = embeddings.embed_documents(documents)

# Generate query embedding
query_vector = embeddings.embed_query(
    "علائم و ویژگی‌های بیماری دیابت چیست؟"
)

print("Document count:", len(document_vectors))
print("Document dimensions:", len(document_vectors[0]))

print("Query dimensions:", len(query_vector))

print(
    "Document vector type:",
    type(document_vectors[0])
)

print(
    "Query vector type:",
    type(query_vector)
)

Document count: 3
Document dimensions: 1024
Query dimensions: 1024
Document vector type: <class 'list'>
Query vector type: <class 'list'>


In [10]:
import numpy as np

documents = [
    "Diabetes is a chronic disease that affects blood sugar levels.",
    "دیابت یک بیماری مزمن است که سطح قند خون را تحت تأثیر قرار می‌دهد.",
    "Common symptoms of diabetes include increased thirst and frequent urination.",
    "علائم شایع دیابت شامل تشنگی زیاد و تکرر ادرار هستند.",
    "Hypertension is a condition characterized by high blood pressure.",
    "فشار خون بالا یکی از عوامل خطر بیماری‌های قلبی است.",
    "Regular exercise improves cardiovascular health.",
    "ورزش منظم به بهبود سلامت قلب و عروق کمک می‌کند.",
]

query = "علائم بیماری دیابت چیست؟"

# Generate passage embeddings
document_vectors = embeddings.embed_documents(documents)

# Generate query embedding
query_vector = embeddings.embed_query(query)

# Calculate similarity
results = []

for document, vector in zip(documents, document_vectors):
    score = float(
        np.dot(
            np.array(query_vector),
            np.array(vector)
        )
    )

    results.append({
        "document": document,
        "similarity": score,
    })

# Rank results
results.sort(
    key=lambda item: item["similarity"],
    reverse=True
)

# Display top results
print("Query:", query)
print("\nTop Retrieval Results:\n")

for index, result in enumerate(results[:5], start=1):
    print(f"{index}. Similarity: {result['similarity']:.4f}")
    print(f"   {result['document']}\n")

Query: علائم بیماری دیابت چیست؟

Top Retrieval Results:

1. Similarity: 0.7603
   علائم شایع دیابت شامل تشنگی زیاد و تکرر ادرار هستند.

2. Similarity: 0.7146
   Common symptoms of diabetes include increased thirst and frequent urination.

3. Similarity: 0.6146
   Diabetes is a chronic disease that affects blood sugar levels.

4. Similarity: 0.5425
   دیابت یک بیماری مزمن است که سطح قند خون را تحت تأثیر قرار می‌دهد.

5. Similarity: 0.2538
   Hypertension is a condition characterized by high blood pressure.



In [11]:
# Synthetic Knowledge Base Dataset

documents = [
    {
        "id": "KB-001",
        "content": "دیابت یک بیماری مزمن است که باعث افزایش سطح قند خون می‌شود."
    },
    {
        "id": "KB-002",
        "content": "علائم شایع دیابت شامل تشنگی زیاد، تکرر ادرار، خستگی و کاهش وزن غیرعادی هستند."
    },
    {
        "id": "KB-003",
        "content": "Diabetes is a chronic disease that affects blood glucose levels."
    },
    {
        "id": "KB-004",
        "content": "Common symptoms of diabetes include excessive thirst, frequent urination, fatigue, and unexplained weight loss."
    },
    {
        "id": "KB-005",
        "content": "فشار خون بالا یکی از عوامل خطر مهم برای بیماری‌های قلبی و عروقی است."
    },
    {
        "id": "KB-006",
        "content": "Hypertension is a medical condition characterized by persistently elevated blood pressure."
    },
    {
        "id": "KB-007",
        "content": "ورزش منظم، تغذیه سالم و خواب کافی به بهبود سلامت قلب کمک می‌کنند."
    },
    {
        "id": "KB-008",
        "content": "Regular physical activity can improve cardiovascular health and reduce the risk of heart disease."
    },
    {
        "id": "KB-009",
        "content": "آسم یک بیماری مزمن تنفسی است که می‌تواند باعث خس‌خس سینه، سرفه و تنگی نفس شود."
    },
    {
        "id": "KB-010",
        "content": "Asthma is a chronic respiratory condition that can cause wheezing, coughing, and shortness of breath."
    },
    {
        "id": "KB-011",
        "content": "کم‌خونی معمولاً با کاهش هموگلوبین خون همراه است و می‌تواند باعث خستگی و ضعف شود."
    },
    {
        "id": "KB-012",
        "content": "Anemia is a condition in which the blood has insufficient healthy red blood cells or hemoglobin."
    },
]

# Evaluation Queries

evaluation_queries = [
    {
        "query": "علائم بیماری دیابت چیست؟",
        "relevant_ids": ["KB-002", "KB-004"]
    },
    {
        "query": "What are the common symptoms of diabetes?",
        "relevant_ids": ["KB-004", "KB-002"]
    },
    {
        "query": "فشار خون بالا چه خطری برای سلامتی دارد؟",
        "relevant_ids": ["KB-005", "KB-006"]
    },
    {
        "query": "How can I improve my heart health?",
        "relevant_ids": ["KB-008", "KB-007"]
    },
    {
        "query": "علائم بیماری آسم چیست؟",
        "relevant_ids": ["KB-009", "KB-010"]
    },
    {
        "query": "What causes anemia-related fatigue?",
        "relevant_ids": ["KB-012", "KB-011"]
    },
    {
        "query": "دیابت چگونه بر قند خون تأثیر می‌گذارد؟",
        "relevant_ids": ["KB-001", "KB-003"]
    },
    {
        "query": "What is hypertension?",
        "relevant_ids": ["KB-006", "KB-005"]
    },
]

print(f"Documents: {len(documents)}")
print(f"Evaluation queries: {len(evaluation_queries)}")

for item in documents:
    print(item["id"], ":", item["content"])

Documents: 12
Evaluation queries: 8
KB-001 : دیابت یک بیماری مزمن است که باعث افزایش سطح قند خون می‌شود.
KB-002 : علائم شایع دیابت شامل تشنگی زیاد، تکرر ادرار، خستگی و کاهش وزن غیرعادی هستند.
KB-003 : Diabetes is a chronic disease that affects blood glucose levels.
KB-004 : Common symptoms of diabetes include excessive thirst, frequent urination, fatigue, and unexplained weight loss.
KB-005 : فشار خون بالا یکی از عوامل خطر مهم برای بیماری‌های قلبی و عروقی است.
KB-006 : Hypertension is a medical condition characterized by persistently elevated blood pressure.
KB-007 : ورزش منظم، تغذیه سالم و خواب کافی به بهبود سلامت قلب کمک می‌کنند.
KB-008 : Regular physical activity can improve cardiovascular health and reduce the risk of heart disease.
KB-009 : آسم یک بیماری مزمن تنفسی است که می‌تواند باعث خس‌خس سینه، سرفه و تنگی نفس شود.
KB-010 : Asthma is a chronic respiratory condition that can cause wheezing, coughing, and shortness of breath.
KB-011 : کم‌خونی معمولاً با کاهش هموگلوبین خون همراه ا

In [16]:
import numpy as np


# 1. Generate document embeddings
document_texts = [doc["content"] for doc in documents]
document_ids = [doc["id"] for doc in documents]

document_vectors = embeddings.embed_documents(document_texts)

print("Document embeddings:", len(document_vectors))
print("Vector dimensions:", len(document_vectors[0]))

# 2. Retrieval evaluation
evaluation_results = []

for item in evaluation_queries:
    query = item["query"]
    relevant_ids = item["relevant_ids"]

    query_vector = np.array(embeddings.embed_query(query))

    scores = []

    for doc, vector in zip(documents, document_vectors):
        similarity = float(
            np.dot(query_vector, np.array(vector))
        )

        scores.append({
            "document_id": doc["id"],
            "similarity": similarity
        })

    ranked_results = sorted(
        scores,
        key=lambda x: x["similarity"],
        reverse=True
    )

    top_3_ids = [
        result["document_id"]
        for result in ranked_results[:3]
    ]

    # Recall@3
    retrieved_relevant = len(
        set(top_3_ids) & set(relevant_ids)
    )

    recall_at_3 = retrieved_relevant / len(relevant_ids)

    # Reciprocal Rank (first relevant result)
    reciprocal_rank = 0.0

    for rank, result in enumerate(ranked_results, start=1):
        if result["document_id"] in relevant_ids:
            reciprocal_rank = 1 / rank
            break

    evaluation_results.append({
        "query": query,
        "top_3": top_3_ids,
        "relevant_ids": relevant_ids,
        "recall_at_3": recall_at_3,
        "reciprocal_rank": reciprocal_rank
    })

# 3. Calculate final metrics
# Final metrics without pandas

mean_recall_at_3 = sum(
    item["recall_at_3"] for item in evaluation_results
) / len(evaluation_results)

mean_reciprocal_rank = sum(
    item["reciprocal_rank"] for item in evaluation_results
) / len(evaluation_results)

print("\nEvaluation Results")
print("-----------------------------")

for item in evaluation_results:
    print(f"\nQuery: {item['query']}")
    print(f"Expected: {item['relevant_ids']}")
    print(f"Retrieved: {item['top_3']}")
    print(f"Recall@3: {item['recall_at_3']:.2f}")
    print(f"Reciprocal Rank: {item['reciprocal_rank']:.2f}")

print("\nFinal Metrics")
print("-----------------------------")
print(f"Mean Recall@3: {mean_recall_at_3:.4f}")
print(f"Mean Reciprocal Rank (MRR): {mean_reciprocal_rank:.4f}")

Document embeddings: 12
Vector dimensions: 1024

Evaluation Results
-----------------------------

Query: علائم بیماری دیابت چیست؟
Expected: ['KB-002', 'KB-004']
Retrieved: ['KB-002', 'KB-004', 'KB-003']
Recall@3: 1.00
Reciprocal Rank: 1.00

Query: What are the common symptoms of diabetes?
Expected: ['KB-004', 'KB-002']
Retrieved: ['KB-004', 'KB-002', 'KB-003']
Recall@3: 1.00
Reciprocal Rank: 1.00

Query: فشار خون بالا چه خطری برای سلامتی دارد؟
Expected: ['KB-005', 'KB-006']
Retrieved: ['KB-005', 'KB-006', 'KB-001']
Recall@3: 1.00
Reciprocal Rank: 1.00

Query: How can I improve my heart health?
Expected: ['KB-008', 'KB-007']
Retrieved: ['KB-008', 'KB-007', 'KB-005']
Recall@3: 1.00
Reciprocal Rank: 1.00

Query: علائم بیماری آسم چیست؟
Expected: ['KB-009', 'KB-010']
Retrieved: ['KB-009', 'KB-010', 'KB-002']
Recall@3: 1.00
Reciprocal Rank: 1.00

Query: What causes anemia-related fatigue?
Expected: ['KB-012', 'KB-011']
Retrieved: ['KB-012', 'KB-011', 'KB-004']
Recall@3: 1.00
Reciprocal Rank

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("LangChain Text Splitters imported successfully!")

LangChain Text Splitters imported successfully!


In [18]:
long_documents = [
    {
        "id": "LONG-001",
        "language": "fa",
        "content": """
        دیابت یک بیماری مزمن است که بر نحوه استفاده بدن از گلوکز تأثیر می‌گذارد.
        گلوکز یکی از منابع اصلی انرژی بدن محسوب می‌شود و انسولین به انتقال آن از خون به سلول‌ها کمک می‌کند.
        در افراد مبتلا به دیابت، بدن ممکن است انسولین کافی تولید نکند یا نتواند به‌درستی از آن استفاده کند.
        
        علائم شایع دیابت شامل تشنگی زیاد، تکرر ادرار، خستگی و کاهش وزن غیرعادی هستند.
        برخی افراد ممکن است در مراحل اولیه بیماری علائم واضحی نداشته باشند.
        تشخیص دیابت معمولاً با آزمایش‌های مرتبط با قند خون انجام می‌شود.
        
        مدیریت دیابت می‌تواند شامل برنامه غذایی مناسب، فعالیت بدنی منظم و پایش قند خون باشد.
        برنامه درمانی باید توسط پزشک و بر اساس شرایط هر فرد تعیین شود.
        آموزش بیمار و پیگیری منظم نیز در مدیریت بلندمدت بیماری اهمیت دارند.
        """
    },
    {
        "id": "LONG-002",
        "language": "en",
        "content": """
        Diabetes is a chronic condition that affects how the body processes glucose.
        Glucose is an important source of energy, and insulin helps move glucose from the bloodstream into cells.
        In people with diabetes, the body may not produce enough insulin or may not use insulin effectively.
        
        Common symptoms of diabetes include excessive thirst, frequent urination, fatigue, and unexplained weight loss.
        Some people may have no noticeable symptoms during the early stages of the condition.
        Diabetes is commonly identified through blood glucose testing.
        
        Diabetes management may involve a balanced diet, regular physical activity, and blood glucose monitoring.
        A healthcare professional should determine an appropriate treatment plan for each individual.
        Patient education and regular follow-up are also important for long-term management.
        """
    },
    {
        "id": "LONG-003",
        "language": "fa",
        "content": """
        فشار خون بالا به وضعیتی گفته می‌شود که فشار واردشده توسط خون به دیواره رگ‌ها به‌طور مداوم افزایش یافته باشد.
        این وضعیت ممکن است برای مدت طولانی بدون علامت مشخص باقی بماند.
        اندازه‌گیری منظم فشار خون برای شناسایی و پیگیری این وضعیت اهمیت دارد.
        
        فشار خون بالا می‌تواند خطر ابتلا به بیماری‌های قلبی، سکته مغزی و مشکلات کلیوی را افزایش دهد.
        عوامل مختلفی مانند سابقه خانوادگی، تغذیه نامناسب، کم‌تحرکی و برخی بیماری‌ها می‌توانند در بروز آن نقش داشته باشند.
        
        تغییر سبک زندگی ممکن است به کنترل فشار خون کمک کند.
        کاهش مصرف نمک، فعالیت بدنی منظم و پیگیری پزشکی از اقدامات مهم در مدیریت فشار خون هستند.
        """
    },
    {
        "id": "LONG-004",
        "language": "en",
        "content": """
        Hypertension is a condition in which blood consistently exerts increased pressure against artery walls.
        It may remain unnoticed for a long time because many people do not experience obvious symptoms.
        Regular blood pressure measurements are important for identifying and monitoring this condition.
        
        Hypertension can increase the risk of heart disease, stroke, and kidney problems.
        Several factors may contribute to high blood pressure, including family history, unhealthy eating habits,
        physical inactivity, and certain medical conditions.
        
        Lifestyle changes may help manage blood pressure.
        Reducing sodium intake, engaging in regular physical activity, and attending medical follow-up appointments
        are important parts of hypertension management.
        """
    }
]

print(f"Long documents created: {len(long_documents)}")

for doc in long_documents:
    print(
        f"{doc['id']} | Language: {doc['language']} | "
        f"Characters: {len(doc['content'])}"
    )

Long documents created: 4
LONG-001 | Language: fa | Characters: 783
LONG-002 | Language: en | Characters: 913
LONG-003 | Language: fa | Characters: 656
LONG-004 | Language: en | Characters: 827


In [19]:
chunk_configs = [
    {"name": "A", "chunk_size": 500, "chunk_overlap": 75},
    {"name": "B", "chunk_size": 800, "chunk_overlap": 120},
    {"name": "C", "chunk_size": 1000, "chunk_overlap": 150},
]

chunking_results = {}

for config in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=[
            "\n\n",
            "\n",
            "؟",
            ".",
            "؛",
            "،",
            " ",
            ""
        ],
    )

    all_chunks = []

    for doc in long_documents:
        chunks = splitter.split_text(doc["content"])

        for index, chunk in enumerate(chunks):
            all_chunks.append({
                "document_id": doc["id"],
                "language": doc["language"],
                "chunk_index": index,
                "content": chunk,
                "characters": len(chunk),
            })

    chunking_results[config["name"]] = all_chunks

    print(f"\nConfiguration {config['name']}")
    print("-" * 40)
    print(f"Chunk size: {config['chunk_size']}")
    print(f"Overlap: {config['chunk_overlap']}")
    print(f"Total chunks: {len(all_chunks)}")

    for doc in long_documents:
        doc_chunks = [
            chunk for chunk in all_chunks
            if chunk["document_id"] == doc["id"]
        ]

        print(f"{doc['id']}: {len(doc_chunks)} chunks")


Configuration A
----------------------------------------
Chunk size: 500
Overlap: 75
Total chunks: 10
LONG-001: 2 chunks
LONG-002: 3 chunks
LONG-003: 2 chunks
LONG-004: 3 chunks

Configuration B
----------------------------------------
Chunk size: 800
Overlap: 120
Total chunks: 6
LONG-001: 1 chunks
LONG-002: 2 chunks
LONG-003: 1 chunks
LONG-004: 2 chunks

Configuration C
----------------------------------------
Chunk size: 1000
Overlap: 150
Total chunks: 4
LONG-001: 1 chunks
LONG-002: 1 chunks
LONG-003: 1 chunks
LONG-004: 1 chunks


In [20]:
import numpy as np

chunking_queries = [
    {
        "query": "علائم شایع بیماری دیابت چیست؟",
        "relevant_ids": ["LONG-001", "LONG-002"]
    },
    {
        "query": "What are the symptoms of diabetes?",
        "relevant_ids": ["LONG-001", "LONG-002"]
    },
    {
        "query": "فشار خون بالا چه خطراتی دارد؟",
        "relevant_ids": ["LONG-003", "LONG-004"]
    },
    {
        "query": "What are the risks of hypertension?",
        "relevant_ids": ["LONG-003", "LONG-004"]
    },
    {
        "query": "چگونه می‌توان دیابت را مدیریت کرد؟",
        "relevant_ids": ["LONG-001", "LONG-002"]
    },
    {
        "query": "How can blood pressure be managed?",
        "relevant_ids": ["LONG-003", "LONG-004"]
    },
]

chunking_evaluation = {}

for config_name, chunks in chunking_results.items():

    # Generate embeddings for all chunks
    chunk_texts = [chunk["content"] for chunk in chunks]
    chunk_vectors = embeddings.embed_documents(chunk_texts)

    recalls = []
    reciprocal_ranks = []

    for item in chunking_queries:
        query_vector = np.array(
            embeddings.embed_query(item["query"])
        )

        # Keep the highest similarity for each document.
        # This prevents multiple chunks from one document
        # occupying several positions in the ranking.
        document_scores = {}

        for chunk, vector in zip(chunks, chunk_vectors):
            score = float(
                np.dot(query_vector, np.array(vector))
            )

            doc_id = chunk["document_id"]

            document_scores[doc_id] = max(
                document_scores.get(doc_id, -1.0),
                score
            )

        ranked_documents = sorted(
            document_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        top_3_ids = [
            doc_id for doc_id, score in ranked_documents[:3]
        ]

        relevant_ids = item["relevant_ids"]

        recall = (
            len(set(top_3_ids) & set(relevant_ids))
            / len(relevant_ids)
        )

        reciprocal_rank = 0.0

        for rank, (doc_id, score) in enumerate(
            ranked_documents, start=1
        ):
            if doc_id in relevant_ids:
                reciprocal_rank = 1 / rank
                break

        recalls.append(recall)
        reciprocal_ranks.append(reciprocal_rank)

        print(f"\nConfiguration: {config_name}")
        print(f"Query: {item['query']}")
        print(f"Expected: {relevant_ids}")
        print(f"Retrieved: {top_3_ids}")
        print(f"Recall@3: {recall:.2f}")
        print(f"Reciprocal Rank: {reciprocal_rank:.2f}")

    chunking_evaluation[config_name] = {
        "total_chunks": len(chunks),
        "mean_recall_at_3": float(np.mean(recalls)),
        "mrr": float(np.mean(reciprocal_ranks))
    }

print("\n\nFINAL CHUNKING COMPARISON")
print("=" * 55)

for name, metrics in chunking_evaluation.items():
    print(f"\nConfiguration {name}")
    print(f"Total chunks: {metrics['total_chunks']}")
    print(f"Mean Recall@3: {metrics['mean_recall_at_3']:.4f}")
    print(f"MRR: {metrics['mrr']:.4f}")


Configuration: A
Query: علائم شایع بیماری دیابت چیست؟
Expected: ['LONG-001', 'LONG-002']
Retrieved: ['LONG-001', 'LONG-002', 'LONG-003']
Recall@3: 1.00
Reciprocal Rank: 1.00

Configuration: A
Query: What are the symptoms of diabetes?
Expected: ['LONG-001', 'LONG-002']
Retrieved: ['LONG-002', 'LONG-001', 'LONG-004']
Recall@3: 1.00
Reciprocal Rank: 1.00

Configuration: A
Query: فشار خون بالا چه خطراتی دارد؟
Expected: ['LONG-003', 'LONG-004']
Retrieved: ['LONG-003', 'LONG-004', 'LONG-001']
Recall@3: 1.00
Reciprocal Rank: 1.00

Configuration: A
Query: What are the risks of hypertension?
Expected: ['LONG-003', 'LONG-004']
Retrieved: ['LONG-004', 'LONG-003', 'LONG-002']
Recall@3: 1.00
Reciprocal Rank: 1.00

Configuration: A
Query: چگونه می‌توان دیابت را مدیریت کرد؟
Expected: ['LONG-001', 'LONG-002']
Retrieved: ['LONG-002', 'LONG-001', 'LONG-003']
Recall@3: 1.00
Reciprocal Rank: 1.00

Configuration: A
Query: How can blood pressure be managed?
Expected: ['LONG-003', 'LONG-004']
Retrieved: ['L

In [21]:
advanced_documents = [
    {
        "id": "ADV-001",
        "topic": "diabetes",
        "language": "fa",
        "content": """
# دیابت و عوامل مؤثر بر آن

دیابت یکی از بیماری‌های مزمن متابولیک است که بر نحوه تنظیم قند خون تأثیر می‌گذارد. گلوکز منبع مهم انرژی برای سلول‌های بدن است و هورمون انسولین نقش مهمی در انتقال آن از جریان خون به سلول‌ها دارد. در افراد مبتلا به دیابت، تولید انسولین یا نحوه استفاده بدن از آن ممکن است دچار اختلال شود. در نتیجه، سطح قند خون می‌تواند برای مدت طولانی بالاتر از محدوده طبیعی باقی بماند.

## علائم و نشانه‌های دیابت

تشنگی بیش از حد، تکرر ادرار، خستگی مداوم، تاری دید و کاهش وزن غیرمنتظره از علائمی هستند که ممکن است در افراد مبتلا به دیابت مشاهده شوند. شدت علائم در افراد مختلف یکسان نیست و برخی افراد ممکن است در مراحل اولیه هیچ نشانه مشخصی نداشته باشند. وجود یک یا چند علامت به‌تنهایی برای تشخیص بیماری کافی نیست.

## عوامل خطر و پیشگیری

سابقه خانوادگی، اضافه وزن، کم‌تحرکی و برخی عوامل مرتبط با سبک زندگی می‌توانند احتمال ابتلا به بعضی انواع دیابت را افزایش دهند. تغذیه متعادل، فعالیت بدنی منظم و حفظ وزن مناسب از اقدامات مفید برای سلامت متابولیک هستند. با این حال، همه عوامل خطر قابل کنترل نیستند و سابقه خانوادگی نیز اهمیت دارد.

## تشخیص و پایش

تشخیص دیابت با استفاده از آزمایش‌های معتبر قند خون و ارزیابی پزشکی انجام می‌شود. پزشک ممکن است بر اساس شرایط فرد از آزمایش قند خون ناشتا یا آزمایش HbA1c استفاده کند. پایش منظم وضعیت قند خون می‌تواند به ارزیابی روند بیماری کمک کند.

## مدیریت و مراقبت بلندمدت

مدیریت دیابت معمولاً به یک برنامه فردی نیاز دارد که می‌تواند شامل تغذیه مناسب، فعالیت بدنی، پایش قند خون و درمان تحت نظر پزشک باشد. آموزش بیمار و آگاهی از علائم احتمالی افت یا افزایش قند خون نیز اهمیت دارد. برنامه مراقبتی باید بر اساس شرایط و نیازهای هر فرد تنظیم شود.
"""
    },
    {
        "id": "ADV-002",
        "topic": "diabetes",
        "language": "en",
        "content": """
# Understanding Diabetes and Its Effects

Diabetes is a chronic metabolic condition that affects how the body regulates blood glucose. Glucose is an important source of energy for cells, while insulin helps move glucose from the bloodstream into body tissues. In people with diabetes, insulin production or insulin effectiveness may be impaired. As a result, blood glucose levels can remain elevated over time.

## Symptoms and Warning Signs

Common symptoms of diabetes may include excessive thirst, frequent urination, persistent fatigue, blurred vision, and unexplained weight changes. Symptoms can vary considerably between individuals. Some people may have very mild symptoms or may not notice any obvious changes during the early stages. Symptoms alone cannot establish a diagnosis.

## Risk Factors and Prevention

Family history, excess body weight, physical inactivity, and certain lifestyle factors may contribute to the risk of developing some forms of diabetes. Balanced nutrition, regular physical activity, and maintaining a healthy weight can support metabolic health. However, not every risk factor is preventable, and inherited factors may also play a role.

## Diagnosis and Monitoring

Diabetes is evaluated through appropriate blood glucose tests and clinical assessment. Healthcare professionals may use fasting blood glucose measurements or the HbA1c test depending on the situation. Regular monitoring can help assess changes in glucose control over time and support informed healthcare decisions.

## Long-Term Management

Diabetes management requires an individualized care plan. Depending on the person, this may involve nutrition planning, physical activity, glucose monitoring, and medical treatment prescribed by a healthcare professional. Patient education and awareness of possible signs of abnormal blood glucose are also important parts of long-term care.
"""
    },
    {
        "id": "ADV-003",
        "topic": "hypertension",
        "language": "fa",
        "content": """
# فشار خون بالا و سلامت عروق

فشار خون بالا وضعیتی است که در آن نیروی واردشده توسط خون به دیواره شریان‌ها به‌طور مداوم افزایش می‌یابد. فشار خون با دو عدد اندازه‌گیری می‌شود که نشان‌دهنده فشار هنگام انقباض و استراحت قلب هستند. اندازه‌گیری صحیح و تکرارشونده برای ارزیابی وضعیت فشار خون اهمیت دارد.

## علائم و نشانه‌ها

فشار خون بالا در بسیاری از افراد بدون علامت مشخص ایجاد می‌شود. به همین دلیل ممکن است فرد برای مدت طولانی از وضعیت خود اطلاع نداشته باشد. سردرد یا سرگیجه به‌تنهایی نشانه قابل اعتمادی برای تشخیص فشار خون بالا نیستند. اندازه‌گیری فشار خون روش اصلی شناسایی این وضعیت محسوب می‌شود.

## عوامل خطر

سابقه خانوادگی، افزایش سن، مصرف زیاد نمک، کم‌تحرکی، مصرف دخانیات و برخی بیماری‌های زمینه‌ای می‌توانند با افزایش خطر فشار خون بالا ارتباط داشته باشند. عوامل مختلف ممکن است هم‌زمان بر وضعیت عروق و عملکرد قلب تأثیر بگذارند. بررسی سابقه پزشکی و سبک زندگی در ارزیابی خطر اهمیت دارد.

## عوارض احتمالی

فشار خون کنترل‌نشده می‌تواند به دیواره عروق آسیب وارد کند و خطر برخی مشکلات قلبی، سکته مغزی و بیماری‌های کلیوی را افزایش دهد. این عوارض معمولاً در ارتباط با فشار خون بالا در طول زمان مطرح می‌شوند. کنترل و پیگیری منظم می‌تواند در کاهش خطر عوارض نقش داشته باشد.

## پایش و مدیریت

اندازه‌گیری منظم فشار خون، تغذیه متعادل، کاهش مصرف نمک، فعالیت بدنی و پیگیری پزشکی از اجزای مهم مراقبت هستند. بعضی افراد ممکن است به درمان دارویی نیاز داشته باشند که انتخاب آن بر عهده پزشک است. تصمیم‌گیری درباره درمان باید بر اساس وضعیت عمومی و عوامل خطر هر فرد انجام شود.
"""
    },
    {
        "id": "ADV-004",
        "topic": "hypertension",
        "language": "en",
        "content": """
# Hypertension and Vascular Health

Hypertension is a condition in which blood consistently exerts increased pressure against artery walls. Blood pressure measurements contain two values representing pressure during heart contraction and relaxation. Accurate measurements taken over time are important when evaluating a person's blood pressure status.

## Symptoms and Clinical Signs

High blood pressure often develops without noticeable symptoms. Many individuals may have the condition for an extended period without knowing it. Headaches and dizziness alone are not reliable indicators of hypertension. Measuring blood pressure using an appropriate device is the primary way to identify elevated readings.

## Risk Factors

Family history, aging, high sodium intake, physical inactivity, tobacco use, and certain underlying medical conditions may contribute to hypertension risk. Multiple factors can influence vascular health and cardiovascular function at the same time. Reviewing lifestyle patterns and medical history helps healthcare professionals assess individual risk.

## Possible Complications

Uncontrolled hypertension can damage blood vessel walls over time and increase the risk of cardiovascular disease, stroke, and kidney complications. The potential consequences depend on several factors, including the duration and severity of elevated blood pressure. Regular monitoring and appropriate medical care are important.

## Monitoring and Management

Blood pressure monitoring, balanced nutrition, reducing excessive sodium intake, physical activity, and regular healthcare follow-up are important parts of management. Some individuals may require prescribed medication. Treatment decisions should be individualized by a qualified healthcare professional based on the person's overall health and risk factors.
"""
    },
    {
        "id": "ADV-005",
        "topic": "asthma",
        "language": "fa",
        "content": """
# آسم و عملکرد دستگاه تنفسی

آسم یک بیماری مزمن تنفسی است که با التهاب و حساسیت بیش از حد راه‌های هوایی ارتباط دارد. در افراد مبتلا، راه‌های هوایی ممکن است در واکنش به محرک‌های مختلف تنگ شوند. این تغییرات می‌توانند عبور هوا را دشوار کنند و باعث ایجاد علائم تنفسی شوند. شدت و الگوی علائم در افراد مختلف متفاوت است.

## علائم شایع آسم

خس‌خس سینه، سرفه، احساس فشار در قفسه سینه و تنگی نفس از نشانه‌های رایج آسم هستند. علائم ممکن است در شب، هنگام فعالیت بدنی یا پس از تماس با محرک‌های محیطی تشدید شوند. برخی افراد دوره‌هایی از بهبود و بازگشت علائم را تجربه می‌کنند.

## محرک‌های محیطی

گردوغبار، گرده گیاهان، دود سیگار، آلودگی هوا، هوای سرد و عفونت‌های تنفسی می‌توانند در برخی افراد علائم آسم را تحریک کنند. محرک‌ها برای همه بیماران یکسان نیستند. شناسایی عوامل شخصی و کاهش تماس با آن‌ها می‌تواند بخشی از برنامه مراقبتی باشد.

## تشخیص و ارزیابی

ارزیابی آسم شامل بررسی سابقه علائم، معاینه پزشکی و در صورت نیاز آزمایش عملکرد ریه است. آزمایش‌هایی مانند اسپیرومتری می‌توانند به بررسی جریان هوا و عملکرد تنفسی کمک کنند. تشخیص باید توسط متخصص سلامت انجام شود.

## مدیریت و کنترل بیماری

مدیریت آسم ممکن است شامل پرهیز از محرک‌های شناخته‌شده، پیگیری پزشکی و استفاده از درمان تجویزشده باشد. داشتن برنامه مشخص برای کنترل علائم و شناخت نشانه‌های تشدید بیماری اهمیت دارد. تنگی نفس شدید یا بدترشونده نیازمند ارزیابی فوری پزشکی است.
"""
    },
    {
        "id": "ADV-006",
        "topic": "asthma",
        "language": "en",
        "content": """
# Asthma and Respiratory Function

Asthma is a chronic respiratory disease associated with inflammation and increased sensitivity of the airways. In affected individuals, the airways may narrow in response to different triggers. These changes can make airflow more difficult and lead to respiratory symptoms. The severity and pattern of symptoms vary between individuals.

## Common Symptoms

Wheezing, coughing, chest tightness, and shortness of breath are common symptoms of asthma. Symptoms may become more noticeable at night, during physical activity, or after exposure to environmental triggers. Some individuals experience periods when symptoms improve and later return.

## Environmental Triggers

Dust, pollen, tobacco smoke, air pollution, cold air, and respiratory infections may trigger asthma symptoms in some people. Triggers are not identical for every patient. Identifying personal triggers and reducing exposure to them can be an important part of a respiratory care plan.

## Diagnosis and Assessment

Asthma assessment includes reviewing symptom history, performing a clinical examination, and sometimes conducting lung function tests. Spirometry can help evaluate airflow and respiratory function. A healthcare professional interprets the findings together with the patient's history before making a diagnosis.

## Disease Management

Asthma management may involve avoiding known triggers, attending medical follow-up appointments, and using prescribed treatment. A personalized action plan can help individuals understand how to monitor symptoms and respond when their condition changes. Severe or worsening breathing difficulty requires urgent medical assessment.
"""
    },
    {
        "id": "ADV-007",
        "topic": "anemia",
        "language": "fa",
        "content": """
# کم‌خونی و انتقال اکسیژن

کم‌خونی وضعیتی است که در آن مقدار هموگلوبین یا تعداد گلبول‌های قرمز سالم برای تأمین نیازهای بدن کافی نیست. هموگلوبین پروتئینی در گلبول‌های قرمز است که وظیفه انتقال اکسیژن به بافت‌ها را بر عهده دارد. کاهش ظرفیت انتقال اکسیژن می‌تواند بر عملکرد طبیعی اندام‌های مختلف تأثیر بگذارد.

## علائم و نشانه‌ها

خستگی، ضعف، رنگ‌پریدگی، سرگیجه، سردرد و کاهش تحمل فعالیت بدنی از علائم احتمالی کم‌خونی هستند. شدت نشانه‌ها به علت کم‌خونی و میزان کاهش هموگلوبین بستگی دارد. بعضی افراد ممکن است علائم خفیفی داشته باشند و برخی دیگر با محدودیت بیشتری در فعالیت‌های روزانه روبه‌رو شوند.

## علل احتمالی

کمبود آهن یکی از علل شناخته‌شده کم‌خونی است، اما کمبود ویتامین B12 یا فولات، برخی بیماری‌های مزمن و اختلالات ارثی خون نیز می‌توانند نقش داشته باشند. خونریزی طولانی‌مدت یا اختلال در تولید گلبول‌های قرمز نیز از عوامل قابل بررسی هستند.

## تشخیص و آزمایش‌ها

آزمایش شمارش کامل سلول‌های خون یا CBC یکی از بررسی‌های اولیه برای ارزیابی کم‌خونی است. پزشک ممکن است بر اساس نتایج، آزمایش‌های تکمیلی مانند بررسی ذخایر آهن یا ویتامین‌ها را درخواست کند. تعیین علت زمینه‌ای برای انتخاب روش مراقبت مناسب ضروری است.

## مراقبت و پیگیری

روش مدیریت کم‌خونی به علت اصلی آن وابسته است. اصلاح تغذیه، بررسی بیماری‌های زمینه‌ای و درمان‌های تجویزشده ممکن است در برنامه مراقبت قرار بگیرند. مصرف مکمل‌ها بدون ارزیابی علت کم‌خونی همیشه مناسب نیست و باید با نظر متخصص انجام شود.
"""
    },
    {
        "id": "ADV-008",
        "topic": "anemia",
        "language": "en",
        "content": """
# Anemia and Oxygen Transport

Anemia is a condition in which the amount of hemoglobin or the number of healthy red blood cells is insufficient to meet the body's needs. Hemoglobin is a protein inside red blood cells that transports oxygen to body tissues. Reduced oxygen-carrying capacity can affect the normal function of different organs.

## Symptoms and Clinical Features

Fatigue, weakness, pale skin, dizziness, headaches, and reduced exercise tolerance are possible symptoms of anemia. The severity of symptoms depends on the underlying cause and the degree of hemoglobin reduction. Some individuals experience mild symptoms, while others may have more noticeable limitations in daily activities.

## Potential Causes

Iron deficiency is a well-known cause of anemia, but vitamin B12 or folate deficiency, chronic diseases, and inherited blood disorders may also contribute. Long-term blood loss and problems with red blood cell production are additional factors that healthcare professionals may investigate.

## Diagnosis and Laboratory Tests

A complete blood count, commonly called CBC, is one of the initial tests used to evaluate anemia. Depending on the findings, additional tests may assess iron stores, vitamin levels, or other blood characteristics. Identifying the underlying cause is important before selecting an appropriate care approach.

## Care and Follow-up

Anemia management depends on its underlying cause. Nutrition changes, evaluation of associated medical conditions, and prescribed treatment may be included in an individualized plan. Taking supplements without understanding the cause is not always appropriate and should be discussed with a healthcare professional.
"""
    },
    {
        "id": "ADV-009",
        "topic": "cardiovascular",
        "language": "fa",
        "content": """
# سلامت قلب و عروق

سیستم قلبی عروقی شامل قلب و شبکه‌ای از رگ‌های خونی است که خون را در سراسر بدن جابه‌جا می‌کنند. قلب با انقباض منظم خود خون را به اندام‌ها و بافت‌ها می‌رساند. عملکرد مناسب این سیستم برای انتقال اکسیژن و مواد مغذی و حفظ سلامت عمومی ضروری است.

## عوامل مؤثر بر سلامت قلب

فشار خون بالا، کلسترول نامناسب، مصرف دخانیات، کم‌تحرکی و برخی بیماری‌های متابولیک می‌توانند خطر مشکلات قلبی عروقی را افزایش دهند. سابقه خانوادگی و سن نیز در ارزیابی خطر اهمیت دارند. ترکیب چند عامل خطر ممکن است احتمال بروز بیماری را بیشتر کند.

## نشانه‌های هشداردهنده

درد یا فشار قفسه سینه، تنگی نفس غیرعادی، تعریق سرد و احساس ضعف ناگهانی می‌توانند از نشانه‌های مشکلات جدی قلبی باشند. علائم در افراد مختلف ممکن است متفاوت باشند. در صورت بروز علائم شدید یا ناگهانی، ارزیابی اورژانسی ضروری است.

## فعالیت بدنی و تغذیه

فعالیت بدنی منظم می‌تواند به بهبود آمادگی قلبی عروقی کمک کند. تغذیه متعادل شامل سبزیجات، میوه‌ها، غلات کامل و منابع غذایی مناسب نیز برای سلامت قلب اهمیت دارد. نوع و میزان فعالیت باید متناسب با وضعیت سلامت فرد باشد.

## غربالگری و مراقبت

بررسی فشار خون، چربی خون و عوامل خطر متابولیک می‌تواند به شناسایی زودهنگام مشکلات کمک کند. برنامه غربالگری باید با توجه به سن، سابقه خانوادگی و وضعیت پزشکی تنظیم شود. پیگیری منظم با متخصص سلامت بخش مهمی از مراقبت قلبی است.
"""
    },
    {
        "id": "ADV-010",
        "topic": "cardiovascular",
        "language": "en",
        "content": """
# Cardiovascular Health

The cardiovascular system consists of the heart and a network of blood vessels that transport blood throughout the body. Through regular contractions, the heart delivers blood to organs and tissues. Proper cardiovascular function is essential for transporting oxygen and nutrients and supporting overall health.

## Factors Affecting Heart Health

High blood pressure, unhealthy cholesterol levels, tobacco use, physical inactivity, and certain metabolic conditions may increase cardiovascular risk. Age and family history are also relevant when assessing an individual's overall risk. Several risk factors may interact and contribute to the development of cardiovascular disease.

## Warning Signs

Chest pain or pressure, unusual shortness of breath, cold sweating, and sudden weakness may indicate serious cardiovascular problems. Symptoms can vary between individuals and may not always follow a typical pattern. Severe or sudden symptoms require urgent medical assessment.

## Physical Activity and Nutrition

Regular physical activity can improve cardiovascular fitness and support general health. A balanced eating pattern that includes vegetables, fruits, whole grains, and suitable sources of protein can also contribute to heart health. Exercise plans should reflect the person's health status and physical abilities.

## Screening and Prevention

Monitoring blood pressure, cholesterol levels, and metabolic risk factors can help identify potential problems earlier. Screening recommendations depend on age, family history, and individual medical circumstances. Regular healthcare follow-up supports prevention and long-term cardiovascular care.
"""
    },
    {
        "id": "ADV-011",
        "topic": "kidney",
        "language": "fa",
        "content": """
# سلامت کلیه و عملکرد آن

کلیه‌ها دو اندام مهم هستند که خون را تصفیه می‌کنند و به تنظیم تعادل آب و مواد معدنی بدن کمک می‌کنند. آن‌ها در دفع مواد زائد از طریق ادرار نقش دارند و در تنظیم فشار خون نیز مؤثر هستند. عملکرد طبیعی کلیه‌ها برای حفظ تعادل داخلی بدن اهمیت زیادی دارد.

## نشانه‌های احتمالی مشکلات کلیوی

برخی بیماری‌های کلیوی در مراحل اولیه ممکن است علامت واضحی نداشته باشند. تغییر در الگوی ادرار، ورم پاها یا صورت، خستگی مداوم و تغییرات فشار خون می‌توانند در برخی شرایط با مشکلات کلیوی همراه باشند. این علائم اختصاصی نیستند و برای تشخیص به بررسی پزشکی نیاز دارند.

## عوامل خطر بیماری کلیوی

دیابت و فشار خون بالا از عوامل مهم خطر بیماری مزمن کلیه محسوب می‌شوند. سابقه خانوادگی، برخی بیماری‌های خودایمنی و مصرف نامناسب بعضی داروها نیز ممکن است در سلامت کلیه نقش داشته باشند. کنترل بیماری‌های زمینه‌ای برای کاهش خطر آسیب کلیوی اهمیت دارد.

## آزمایش و ارزیابی

پزشکان ممکن است برای ارزیابی عملکرد کلیه از آزمایش خون برای بررسی کراتینین و برآورد میزان فیلتراسیون گلومرولی استفاده کنند. آزمایش ادرار نیز می‌تواند وجود پروتئین یا نشانه‌های دیگر آسیب کلیوی را بررسی کند. تفسیر نتایج باید در کنار شرایط بالینی انجام شود.

## مراقبت از کلیه‌ها

کنترل فشار خون و قند خون، رعایت توصیه‌های تغذیه‌ای و پیگیری پزشکی می‌توانند به حفظ سلامت کلیه کمک کنند. نیازهای مایعات و رژیم غذایی در افراد مختلف متفاوت است، به‌خصوص در کسانی که بیماری کلیوی دارند. برنامه مراقبت باید با نظر متخصص تنظیم شود.
"""
    },
    {
        "id": "ADV-012",
        "topic": "kidney",
        "language": "en",
        "content": """
# Kidney Health and Function

The kidneys are two important organs that filter the blood and help regulate the body's water and mineral balance. They remove waste products through urine and contribute to blood pressure regulation. Normal kidney function is essential for maintaining the body's internal balance.

## Possible Signs of Kidney Problems

Some kidney conditions may not cause noticeable symptoms during their early stages. Changes in urination, swelling in the legs or face, persistent fatigue, and changes in blood pressure may occur in certain kidney disorders. These symptoms are not specific to kidney disease and require appropriate medical evaluation.

## Risk Factors for Kidney Disease

Diabetes and high blood pressure are major risk factors for chronic kidney disease. Family history, certain autoimmune conditions, and inappropriate use of some medications may also affect kidney health. Managing underlying health conditions is important for reducing the risk of kidney damage.

## Laboratory Assessment

Healthcare professionals may evaluate kidney function using blood tests that measure creatinine and estimate glomerular filtration rate. Urine testing can identify protein or other signs that may indicate kidney damage. Results should always be interpreted in the context of the person's medical history and clinical condition.

## Protecting Kidney Health

Managing blood pressure and blood glucose, following appropriate nutrition advice, and attending medical follow-up appointments can support kidney health. Fluid and dietary requirements differ between individuals, particularly among people with existing kidney conditions. A personalized care plan should be developed with a qualified healthcare professional.
"""
    }
]

print(f"Total documents: {len(advanced_documents)}")

for doc in advanced_documents:
    print(
        f"{doc['id']} | {doc['topic']} | {doc['language']} | "
        f"{len(doc['content'])} characters"
    )

Total documents: 12
ADV-001 | diabetes | fa | 1592 characters
ADV-002 | diabetes | en | 1890 characters
ADV-003 | hypertension | fa | 1460 characters
ADV-004 | hypertension | en | 1830 characters
ADV-005 | asthma | fa | 1319 characters
ADV-006 | asthma | en | 1687 characters
ADV-007 | anemia | fa | 1365 characters
ADV-008 | anemia | en | 1702 characters
ADV-009 | cardiovascular | fa | 1270 characters
ADV-010 | cardiovascular | en | 1683 characters
ADV-011 | kidney | fa | 1388 characters
ADV-012 | kidney | en | 1748 characters


In [22]:
advanced_queries = [
    {
        "query": "چه علائمی ممکن است نشان‌دهنده ابتلا به دیابت باشند؟",
        "relevant_doc_ids": ["ADV-001", "ADV-002"],
        "relevant_section": "Symptoms and Warning Signs",
        "topic": "diabetes",
        "language": "fa"
    },
    {
        "query": "Which blood tests are used to diagnose diabetes?",
        "relevant_doc_ids": ["ADV-001", "ADV-002"],
        "relevant_section": "Diagnosis and Monitoring",
        "topic": "diabetes",
        "language": "en"
    },
    {
        "query": "آیا فشار خون بالا همیشه علائم مشخصی دارد؟",
        "relevant_doc_ids": ["ADV-003", "ADV-004"],
        "relevant_section": "Symptoms and Clinical Signs",
        "topic": "hypertension",
        "language": "fa"
    },
    {
        "query": "What are the long-term complications of uncontrolled hypertension?",
        "relevant_doc_ids": ["ADV-003", "ADV-004"],
        "relevant_section": "Possible Complications",
        "topic": "hypertension",
        "language": "en"
    },
    {
        "query": "چه عواملی می‌توانند باعث تحریک علائم آسم شوند؟",
        "relevant_doc_ids": ["ADV-005", "ADV-006"],
        "relevant_section": "Environmental Triggers",
        "topic": "asthma",
        "language": "fa"
    },
    {
        "query": "What tests are used to evaluate asthma?",
        "relevant_doc_ids": ["ADV-005", "ADV-006"],
        "relevant_section": "Diagnosis and Assessment",
        "topic": "asthma",
        "language": "en"
    },
    {
        "query": "کدام آزمایش‌ها برای بررسی کم‌خونی استفاده می‌شوند؟",
        "relevant_doc_ids": ["ADV-007", "ADV-008"],
        "relevant_section": "Diagnosis and Laboratory Tests",
        "topic": "anemia",
        "language": "fa"
    },
    {
        "query": "What are common causes of anemia?",
        "relevant_doc_ids": ["ADV-007", "ADV-008"],
        "relevant_section": "Potential Causes",
        "topic": "anemia",
        "language": "en"
    },
    {
        "query": "چه علائمی ممکن است نشان‌دهنده مشکلات جدی قلبی باشند؟",
        "relevant_doc_ids": ["ADV-009", "ADV-010"],
        "relevant_section": "Warning Signs",
        "topic": "cardiovascular",
        "language": "fa"
    },
    {
        "query": "Which lifestyle factors contribute to cardiovascular health?",
        "relevant_doc_ids": ["ADV-009", "ADV-010"],
        "relevant_section": "Physical Activity and Nutrition",
        "topic": "cardiovascular",
        "language": "en"
    },
    {
        "query": "چه بیماری‌هایی خطر ابتلا به بیماری مزمن کلیه را افزایش می‌دهند؟",
        "relevant_doc_ids": ["ADV-011", "ADV-012"],
        "relevant_section": "Risk Factors for Kidney Disease",
        "topic": "kidney",
        "language": "fa"
    },
    {
        "query": "Which laboratory tests help evaluate kidney function?",
        "relevant_doc_ids": ["ADV-011", "ADV-012"],
        "relevant_section": "Laboratory Assessment",
        "topic": "kidney",
        "language": "en"
    },
]

print("Total evaluation queries:", len(advanced_queries))

for index, item in enumerate(advanced_queries, start=1):
    print(
        f"{index}. [{item['language']}] "
        f"{item['query']} | Topic: {item['topic']}"
    )

Total evaluation queries: 12
1. [fa] چه علائمی ممکن است نشان‌دهنده ابتلا به دیابت باشند؟ | Topic: diabetes
2. [en] Which blood tests are used to diagnose diabetes? | Topic: diabetes
3. [fa] آیا فشار خون بالا همیشه علائم مشخصی دارد؟ | Topic: hypertension
4. [en] What are the long-term complications of uncontrolled hypertension? | Topic: hypertension
5. [fa] چه عواملی می‌توانند باعث تحریک علائم آسم شوند؟ | Topic: asthma
6. [en] What tests are used to evaluate asthma? | Topic: asthma
7. [fa] کدام آزمایش‌ها برای بررسی کم‌خونی استفاده می‌شوند؟ | Topic: anemia
8. [en] What are common causes of anemia? | Topic: anemia
9. [fa] چه علائمی ممکن است نشان‌دهنده مشکلات جدی قلبی باشند؟ | Topic: cardiovascular
10. [en] Which lifestyle factors contribute to cardiovascular health? | Topic: cardiovascular
11. [fa] چه بیماری‌هایی خطر ابتلا به بیماری مزمن کلیه را افزایش می‌دهند؟ | Topic: kidney
12. [en] Which laboratory tests help evaluate kidney function? | Topic: kidney


In [23]:
import re

sectioned_documents = []

for doc in advanced_documents:
    sections = re.split(
        r"(?m)^##\s+",
        doc["content"]
    )

    for section_index, section in enumerate(sections):
        section = section.strip()

        if not section:
            continue

        lines = section.splitlines()
        section_title = lines[0].strip().lstrip("#").strip()

        section_content = "\n".join(lines[1:]).strip()

        if not section_content:
            continue

        sectioned_documents.append({
            "document_id": doc["id"],
            "topic": doc["topic"],
            "language": doc["language"],
            "section_index": section_index,
            "section_title": section_title,
            "content": section_content
        })

print("Total sections:", len(sectioned_documents))

for section in sectioned_documents:
    print(
        f"{section['document_id']} | "
        f"Section {section['section_index']} | "
        f"{section['section_title']} | "
        f"{len(section['content'])} characters"
    )

Total sections: 60
ADV-001 | Section 0 | دیابت و عوامل مؤثر بر آن | 367 characters
ADV-001 | Section 1 | علائم و نشانه‌های دیابت | 299 characters
ADV-001 | Section 2 | عوامل خطر و پیشگیری | 293 characters
ADV-001 | Section 3 | تشخیص و پایش | 230 characters
ADV-001 | Section 4 | مدیریت و مراقبت بلندمدت | 268 characters
ADV-002 | Section 0 | Understanding Diabetes and Its Effects | 368 characters
ADV-002 | Section 1 | Symptoms and Warning Signs | 345 characters
ADV-002 | Section 2 | Risk Factors and Prevention | 352 characters
ADV-002 | Section 3 | Diagnosis and Monitoring | 315 characters
ADV-002 | Section 4 | Long-Term Management | 341 characters
ADV-003 | Section 0 | فشار خون بالا و سلامت عروق | 265 characters
ADV-003 | Section 1 | علائم و نشانه‌ها | 276 characters
ADV-003 | Section 2 | عوامل خطر | 277 characters
ADV-003 | Section 3 | عوارض احتمالی | 259 characters
ADV-003 | Section 4 | پایش و مدیریت | 272 characters
ADV-004 | Section 0 | Hypertension and Vascular Health | 315 charact

In [24]:
# Map each query to its expected section index.
# Section indices:
# 1 = Symptoms
# 2 = Risk Factors / Triggers / Causes
# 3 = Diagnosis / Complications / Laboratory Assessment

expected_section_indices = [
    1,  # Diabetes symptoms
    3,  # Diabetes diagnosis
    1,  # Hypertension symptoms
    3,  # Hypertension complications
    2,  # Asthma triggers
    3,  # Asthma diagnosis
    3,  # Anemia diagnosis
    2,  # Anemia causes
    2,  # Cardiovascular warning signs
    3,  # Cardiovascular physical activity and nutrition
    2,  # Kidney risk factors
    3,  # Kidney laboratory assessment
]

for query, section_index in zip(
    advanced_queries,
    expected_section_indices
):
    query["relevant_section_index"] = section_index

    query["relevant_section_ids"] = [
        f"{doc_id}-S{section_index}"
        for doc_id in query["relevant_doc_ids"]
    ]

for index, query in enumerate(advanced_queries, start=1):
    print(f"\nQuery {index}: {query['query']}")
    print("Relevant sections:", query["relevant_section_ids"])


Query 1: چه علائمی ممکن است نشان‌دهنده ابتلا به دیابت باشند؟
Relevant sections: ['ADV-001-S1', 'ADV-002-S1']

Query 2: Which blood tests are used to diagnose diabetes?
Relevant sections: ['ADV-001-S3', 'ADV-002-S3']

Query 3: آیا فشار خون بالا همیشه علائم مشخصی دارد؟
Relevant sections: ['ADV-003-S1', 'ADV-004-S1']

Query 4: What are the long-term complications of uncontrolled hypertension?
Relevant sections: ['ADV-003-S3', 'ADV-004-S3']

Query 5: چه عواملی می‌توانند باعث تحریک علائم آسم شوند؟
Relevant sections: ['ADV-005-S2', 'ADV-006-S2']

Query 6: What tests are used to evaluate asthma?
Relevant sections: ['ADV-005-S3', 'ADV-006-S3']

Query 7: کدام آزمایش‌ها برای بررسی کم‌خونی استفاده می‌شوند؟
Relevant sections: ['ADV-007-S3', 'ADV-008-S3']

Query 8: What are common causes of anemia?
Relevant sections: ['ADV-007-S2', 'ADV-008-S2']

Query 9: چه علائمی ممکن است نشان‌دهنده مشکلات جدی قلبی باشند؟
Relevant sections: ['ADV-009-S2', 'ADV-010-S2']

Query 10: Which lifestyle factors contribu

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_configs = {
    "A": {"chunk_size": 500, "chunk_overlap": 75},
    "B": {"chunk_size": 800, "chunk_overlap": 120},
    "C": {"chunk_size": 1000, "chunk_overlap": 150},
}

separators = ["\n\n", "\n", "؟", ".", "؛", "،", " ", ""]

advanced_chunking_results = {}

for config_name, config in chunk_configs.items():

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=separators,
        length_function=len,
    )

    chunks = []

    for doc in advanced_documents:
        document_chunks = splitter.split_text(doc["content"])

        for chunk_index, chunk_text in enumerate(document_chunks):
            chunks.append({
                "chunk_id": f"{doc['id']}-C{chunk_index}",
                "document_id": doc["id"],
                "topic": doc["topic"],
                "language": doc["language"],
                "chunk_index": chunk_index,
                "content": chunk_text,
                "char_count": len(chunk_text),
            })

    advanced_chunking_results[config_name] = chunks

    print(
        f"Config {config_name}: "
        f"{len(chunks)} chunks | "
        f"Size={config['chunk_size']} | "
        f"Overlap={config['chunk_overlap']}"
    )

Config A: 58 chunks | Size=500 | Overlap=75
Config B: 33 chunks | Size=800 | Overlap=120
Config C: 27 chunks | Size=1000 | Overlap=150


In [27]:
from difflib import SequenceMatcher

def assign_section_labels(chunk):
    document_sections = sections_by_document[chunk["document_id"]]

    scores = []

    for section in document_sections:
        section_text = section["content"]

        score = SequenceMatcher(
            None,
            chunk["content"],
            section_text
        ).ratio()

        scores.append({
            "section_id": (
                f"{section['document_id']}-S{section['section_index']}"
            ),
            "score": score,
        })

    best_match = max(scores, key=lambda item: item["score"])

    return [best_match["section_id"]]


for config_name, chunks in advanced_chunking_results.items():
    for chunk in chunks:
        chunk["section_ids"] = assign_section_labels(chunk)

    print(f"\nConfig {config_name}")

    for chunk in chunks[:3]:
        print(
            chunk["chunk_id"],
            chunk["section_ids"]
        )


Config A
ADV-001-C0 ['ADV-001-S0']
ADV-001-C1 ['ADV-001-S4']
ADV-001-C2 ['ADV-001-S2']

Config B
ADV-001-C0 ['ADV-001-S0']
ADV-001-C1 ['ADV-001-S2']
ADV-001-C2 ['ADV-001-S4']

Config C
ADV-001-C0 ['ADV-001-S0']
ADV-001-C1 ['ADV-001-S2']
ADV-002-C0 ['ADV-002-S0']


In [28]:
section_chunking_results = {}

for config_name, config in chunk_configs.items():

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=separators,
        length_function=len,
    )

    chunks = []

    for section in sectioned_documents:

        section_chunks = splitter.split_text(
            section["content"]
        )

        for chunk_index, chunk_text in enumerate(section_chunks):

            chunks.append({
                "chunk_id": (
                    f"{section['document_id']}-"
                    f"S{section['section_index']}-"
                    f"C{chunk_index}"
                ),
                "document_id": section["document_id"],
                "topic": section["topic"],
                "language": section["language"],
                "section_index": section["section_index"],
                "section_id": (
                    f"{section['document_id']}-"
                    f"S{section['section_index']}"
                ),
                "section_title": section["section_title"],
                "chunk_index": chunk_index,
                "content": chunk_text,
                "char_count": len(chunk_text),
            })

    section_chunking_results[config_name] = chunks

    print(
        f"Config {config_name}: "
        f"{len(chunks)} chunks"
    )

Config A: 60 chunks
Config B: 60 chunks
Config C: 60 chunks


In [29]:
section_lengths = [
    len(section["content"])
    for section in sectioned_documents
]

print("Total sections:", len(section_lengths))
print("Minimum:", min(section_lengths))
print("Maximum:", max(section_lengths))
print("Average:", round(
    sum(section_lengths) / len(section_lengths), 2
))

Total sections: 60
Minimum: 208
Maximum: 368
Average: 287.28


In [30]:
final_chunks = advanced_chunking_results["B"]

print("Total chunks:", len(final_chunks))

print("\nSample chunk:")
print("ID:", final_chunks[0]["chunk_id"])
print("Topic:", final_chunks[0]["topic"])
print("Language:", final_chunks[0]["language"])
print("Characters:", final_chunks[0]["char_count"])
print("\nContent:")
print(final_chunks[0]["content"])

Total chunks: 33

Sample chunk:
ID: ADV-001-C0
Topic: diabetes
Language: fa
Characters: 748

Content:
# دیابت و عوامل مؤثر بر آن

دیابت یکی از بیماری‌های مزمن متابولیک است که بر نحوه تنظیم قند خون تأثیر می‌گذارد. گلوکز منبع مهم انرژی برای سلول‌های بدن است و هورمون انسولین نقش مهمی در انتقال آن از جریان خون به سلول‌ها دارد. در افراد مبتلا به دیابت، تولید انسولین یا نحوه استفاده بدن از آن ممکن است دچار اختلال شود. در نتیجه، سطح قند خون می‌تواند برای مدت طولانی بالاتر از محدوده طبیعی باقی بماند.

## علائم و نشانه‌های دیابت

تشنگی بیش از حد، تکرر ادرار، خستگی مداوم، تاری دید و کاهش وزن غیرمنتظره از علائمی هستند که ممکن است در افراد مبتلا به دیابت مشاهده شوند. شدت علائم در افراد مختلف یکسان نیست و برخی افراد ممکن است در مراحل اولیه هیچ نشانه مشخصی نداشته باشند. وجود یک یا چند علامت به‌تنهایی برای تشخیص بیماری کافی نیست.

## عوامل خطر و پیشگیری


In [31]:
# Generate embeddings for all document chunks

document_texts = [
    chunk["content"]
    for chunk in final_chunks
]

document_embeddings = embeddings.embed_documents(
    document_texts
)

print("Total embeddings:", len(document_embeddings))
print("Embedding dimensions:", len(document_embeddings[0]))
print("Embedding type:", type(document_embeddings[0]))

Total embeddings: 33
Embedding dimensions: 1024
Embedding type: <class 'list'>


In [32]:
import numpy as np

query_texts = [
    item["query"]
    for item in advanced_queries
]

query_embeddings = embeddings._embed(
    query_texts,
    task="retrieval.query"
)

print("Total query embeddings:", len(query_embeddings))
print("Embedding dimensions:", len(query_embeddings[0]))

Total query embeddings: 12
Embedding dimensions: 1024


In [33]:
document_matrix = np.array(document_embeddings)
query_matrix = np.array(query_embeddings)

similarity_matrix = query_matrix @ document_matrix.T

retrieval_results = {}

for query_index, query in enumerate(advanced_queries):

    scores = similarity_matrix[query_index]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for rank, chunk_index in enumerate(ranked_indices[:5], start=1):

        chunk = final_chunks[chunk_index]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "topic": chunk["topic"],
            "language": chunk["language"],
            "similarity": float(scores[chunk_index]),
            "content": chunk["content"],
        })

    retrieval_results[query_index] = results

    print(f"\nQuery {query_index + 1}: {query['query']}")

    for result in results[:3]:
        print(
            f"Rank {result['rank']} | "
            f"{result['chunk_id']} | "
            f"Similarity: {result['similarity']:.4f}"
        )


Query 1: چه علائمی ممکن است نشان‌دهنده ابتلا به دیابت باشند؟
Rank 1 | ADV-001-C0 | Similarity: 0.7011
Rank 2 | ADV-002-C0 | Similarity: 0.6909
Rank 3 | ADV-001-C1 | Similarity: 0.6192

Query 2: Which blood tests are used to diagnose diabetes?
Rank 1 | ADV-001-C1 | Similarity: 0.5580
Rank 2 | ADV-002-C1 | Similarity: 0.5376
Rank 3 | ADV-001-C0 | Similarity: 0.4921

Query 3: آیا فشار خون بالا همیشه علائم مشخصی دارد؟
Rank 1 | ADV-003-C0 | Similarity: 0.7088
Rank 2 | ADV-004-C0 | Similarity: 0.6653
Rank 3 | ADV-003-C1 | Similarity: 0.6014

Query 4: What are the long-term complications of uncontrolled hypertension?
Rank 1 | ADV-004-C1 | Similarity: 0.7041
Rank 2 | ADV-003-C1 | Similarity: 0.6189
Rank 3 | ADV-004-C0 | Similarity: 0.4742

Query 5: چه عواملی می‌توانند باعث تحریک علائم آسم شوند؟
Rank 1 | ADV-005-C1 | Similarity: 0.7989
Rank 2 | ADV-005-C0 | Similarity: 0.7223
Rank 3 | ADV-006-C1 | Similarity: 0.6122

Query 6: What tests are used to evaluate asthma?
Rank 1 | ADV-006-C1 | Simila

In [35]:
evaluation_rows = []

for query_index, query in enumerate(advanced_queries):

    results = retrieval_results[query_index]
    relevant_docs = set(query["relevant_doc_ids"])

    retrieved_docs = [
        result["document_id"]
        for result in results[:3]
    ]

    relevant_retrieved = [
        doc_id for doc_id in retrieved_docs
        if doc_id in relevant_docs
    ]

    recall_at_3 = (
        len(set(relevant_retrieved)) / len(relevant_docs)
    )

    precision_at_3 = (
        len(relevant_retrieved) / 3
    )

    reciprocal_rank = 0

    for rank, result in enumerate(results, start=1):
        if result["document_id"] in relevant_docs:
            reciprocal_rank = 1 / rank
            break

    evaluation_rows.append({
        "query": query["query"],
        "topic": query["topic"],
        "language": query["language"],
        "Recall@3": recall_at_3,
        "Precision@3": precision_at_3,
        "MRR": reciprocal_rank,
    })


print("RETRIEVAL EVALUATION")
print("=" * 90)

for index, row in enumerate(evaluation_rows, start=1):

    print(f"\nQuery {index}: {row['query']}")
    print(f"Recall@3:     {row['Recall@3']:.4f}")
    print(f"Precision@3:  {row['Precision@3']:.4f}")
    print(f"MRR:          {row['MRR']:.4f}")


mean_recall = sum(
    row["Recall@3"] for row in evaluation_rows
) / len(evaluation_rows)

mean_precision = sum(
    row["Precision@3"] for row in evaluation_rows
) / len(evaluation_rows)

mean_mrr = sum(
    row["MRR"] for row in evaluation_rows
) / len(evaluation_rows)

print("\n" + "=" * 90)
print("OVERALL RESULTS")
print("=" * 90)

print(f"Mean Recall@3:    {mean_recall:.4f}")
print(f"Mean Precision@3: {mean_precision:.4f}")
print(f"Mean MRR:         {mean_mrr:.4f}")

RETRIEVAL EVALUATION

Query 1: چه علائمی ممکن است نشان‌دهنده ابتلا به دیابت باشند؟
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 2: Which blood tests are used to diagnose diabetes?
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 3: آیا فشار خون بالا همیشه علائم مشخصی دارد؟
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 4: What are the long-term complications of uncontrolled hypertension?
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 5: چه عواملی می‌توانند باعث تحریک علائم آسم شوند؟
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 6: What tests are used to evaluate asthma?
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 7: کدام آزمایش‌ها برای بررسی کم‌خونی استفاده می‌شوند؟
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 8: What are common causes of anemia?
Recall@3:     1.0000
Precision@3:  1.0000
MRR:          1.0000

Query 9: چه 

In [36]:
import json

evaluation_summary = {
    "embedding_model": "jina-embeddings-v5-text-small",
    "embedding_dimensions": 1024,
    "chunking": {
        "strategy": "RecursiveCharacterTextSplitter",
        "chunk_size": 800,
        "chunk_overlap": 120,
        "total_chunks": len(final_chunks)
    },
    "dataset": {
        "documents": len(advanced_documents),
        "queries": len(advanced_queries)
    },
    "metrics": {
        "mean_recall_at_3": mean_recall,
        "mean_precision_at_3": mean_precision,
        "mrr": mean_mrr
    },
    "evaluation_status": "initial_benchmark"
}

with open(
    "evaluation_results/rag_retrieval_baseline.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        evaluation_summary,
        file,
        ensure_ascii=False,
        indent=4
    )

print("Evaluation results saved successfully.")

Evaluation results saved successfully.
